In [1]:
import pandas as pd

import json
from pathlib import Path

import pandas as pd
import xml.etree.ElementTree as ET
import os
from tei_convertor import JsonToTeiConverter, TEI_NS  # from our earlier design
pd.set_option('display.max_columns', None)

In [2]:
emlap_metadata = pd.read_csv("/srv/data/tome/tome-corpus/emlap_metadata.csv", index_col=0, sep=";")

In [3]:
source_path = "../data/sents_data_jsons_dicts/"
xml_dest_path = "../data/emlap_corpus_public/emlap_lemmatized_xmls/"
os.makedirs(xml_dest_path, exist_ok=True)
filenames = os.listdir(source_path)

In [4]:
# set up converter once
converter = JsonToTeiConverter(
    insert_page_breaks=True,
    insert_line_breaks=True,
    margin_as_note=False,  # set True if you want margin-only sentences as <note>
)

# register TEI namespace for prettier output
ET.register_namespace("", TEI_NS)

for filename in filenames:
    if not filename.endswith(".json"):
        continue  # skip any non-json files (e.g. .DS_Store, etc.)

    json_path = os.path.join(source_path, filename)

    with open(json_path, "r", encoding="utf-8") as f:
        sents_data = json.load(f)

    # work id from filename, e.g. "100007.json" -> "100007"
    work_id_int = int(filename.partition(".json")[0])
    work_id_str = str(work_id_int)

    # get metadata row for this work
    row = emlap_metadata.loc[emlap_metadata["no."] == work_id_int]
    if row.empty:
        print(f"[WARN] No metadata for work {work_id_int}, skipping.")
        continue

    work_metadata_series = row.iloc[0]
    meta_dict = work_metadata_series.to_dict()

    # apply wrapper → TEI root element
    tei_root = converter.convert_work(work_id_str, sents_data, meta_dict)

    # write as XML
    filepath = os.path.join(xml_dest_path, f"{work_id_str}.xml")
    tree = ET.ElementTree(tei_root)
    tree.write(filepath, encoding="utf-8", xml_declaration=True)

    print(f"[OK] {work_id_str} → {filepath}")


[OK] 100044 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100044.xml
[OK] 100034 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100034.xml
[OK] 100014 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100014.xml
[OK] 100094 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100094.xml
[OK] 100060 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100060.xml
[OK] 100090 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100090.xml
[OK] 100010 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100010.xml
[OK] 100078 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100078.xml
[OK] 100068 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100068.xml
[OK] 100079 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100079.xml
[OK] 100043 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100043.xml
[OK] 100072 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100072.xml
[OK] 100041 → ../data/emlap_corpus_public/emlap_lemmatized_xmls/100041.xml
[OK] 100012 → ../data/eml